# Day 23 — Practice Session · **SOLUTIONS**
### Descriptive Statistics · Statistics & Probability

**Prepared by Srinivasa Sai Chava**  ·  Boston University

---

> **Instructor copy.** Every question is followed by the answer and the reasoning.
> The student copy (`Day23_Practice_Questions.ipynb`) is identical minus the answer blocks.

| Part | Focus | Questions |
|---|---|---|
| A | Predict the output | 8 |
| B | Spot & fix the bug | 4 |
| C | Write the code | 5 |
| D | Challenge | 2 |

---
## Setup — run this first

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.simplefilter("ignore")

rng = np.random.default_rng(23)

# A small, readable dataset we can check by hand
marks = pd.Series([52, 58, 61, 61, 64, 67, 70, 73, 79, 95])

# And a skewed one
pay = pd.Series(np.r_[rng.normal(42000, 6000, 180), rng.normal(300000, 40000, 20)])

print("marks:", marks.tolist())
print("pay  :", len(pay), "salaries")

---
## Setup — run this first

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.simplefilter("ignore")

rng = np.random.default_rng(23)

# A small, readable dataset we can check by hand
marks = pd.Series([52, 58, 61, 61, 64, 67, 70, 73, 79, 95])

# And a skewed one
pay = pd.Series(np.r_[rng.normal(42000, 6000, 180), rng.normal(300000, 40000, 20)])

print("marks:", marks.tolist())
print("pay  :", len(pay), "salaries")

---
# Part A — Predict the Output

**Teaching note:** A4 and A5 are the ddof pair. A8 is the one that changes how they report.

### A1. What are the centre measures?

In [ ]:
m = pd.Series([52, 58, 61, 61, 64, 67, 70, 73, 79, 95])
print(m.mean(), m.median())

*Your prediction:*  

> ### ✅ Answer A1
> ```
> 68.0 65.5
> ```
> **Why:** ten values, so the median is the average of the 5th and 6th — (64 + 67) / 2 = 65.5.
>
> The mean sits **2.5 above** the median, pulled up by the 95. With only one mild outlier the
> gap is small, but it is already pointing the right way.

### A2. What does mode() give?

In [ ]:
a = pd.Series([52, 58, 61, 61, 64])
b = pd.Series([1, 1, 2, 2, 3])
print(a.mode().tolist())
print(b.mode().tolist())
print(type(b.mode()).__name__)

*Your prediction:*  

> ### ✅ Answer A2
> ```
> [61]
> [1, 2]
> Series
> ```
> **Why:** `mode()` must return a **Series**, because data can have two equally common values
> and there would be nothing to return otherwise.
>
> `b.mode()[0]` would give 1 and **silently hide the tie**. Check `len(s.mode())` first.

### A3. Which is bigger?

In [ ]:
m = pd.Series([52, 58, 61, 61, 64, 67, 70, 73, 79, 95])
print(round(m.std(ddof=0), 3))
print(round(m.std(ddof=1), 3))

*Your prediction:*  

> ### ✅ Answer A3
> ```
> 11.619
> 12.247
> ```
> **`ddof=1` is always the larger of the two**, because you divide by a smaller number
> (9 instead of 10).
>
> The gap here is about 5.4%. On a sample of 5 it would be nearer 12%; on a sample of 500 it is
> negligible — which is why the choice matters most on small data.

### A4. Do these agree?

In [ ]:
x = np.array([52., 58., 61., 61., 64.])
print(round(np.std(x), 4))
print(round(pd.Series(x).std(), 4))

*Your prediction:*  

> ### ✅ Answer A4
> ```
> 4.0694
> 4.5497
> ```
> **They disagree, on identical data.** NumPy defaults to `ddof=0` (population); pandas
> defaults to `ddof=1` (sample).
>
> Neither is wrong. NumPy was built for numerical arrays of whatever you have; pandas for
> sampled tabular data, where `ddof=1` is almost always what you want.
>
> **State `ddof` explicitly whenever the number will be reported to anyone.**

### A5. Which ddof should this use?

In [ ]:
# The marks of ALL 60 students enrolled in this class.
# You want to describe the spread of THIS class, nothing more.
class_marks = pd.Series(rng.integers(40, 100, 60))
# ddof=0 or ddof=1?

*Your prediction:*  

> ### ✅ Answer A5
> **`ddof=0`** — this is a **population**, not a sample.
>
> You have every student you care about. There is no larger unseen group you are trying to
> infer about, so there is no bias to correct for.
>
> **The test is not how big the data is — it is whether anything is unseen.** 60 students
> who are *all* the students is a population; 6,000 customers standing in for *all* customers
> is a sample.
>
> In practice `ddof=1` is right far more often, which is why pandas made it the default.

### A6. What sign is the skew?

In [ ]:
right = pd.Series(np.r_[np.full(90, 10.0), np.full(10, 90.0)])   # a few very high
left  = pd.Series(np.r_[np.full(90, 90.0), np.full(10, 10.0)])   # a few very low
print(round(right.skew(), 2))
print(round(left.skew(), 2))

*Your prediction:*  

> ### ✅ Answer A6
> ```
> 2.71
> -2.71
> ```
> **Positive skew = long tail to the RIGHT.** The sign tells you which side the tail is on,
> not which side most of the data is on — those are opposites, which is the usual confusion.
>
> `right` has most values at 10 with a few at 90, so the tail stretches right → positive.
>
> Rough guide: `|skew| > 1` means strongly skewed, so **report the median**.

### A7. Same mean, same SD?

In [ ]:
one = pd.Series(np.r_[np.full(100, 40.0), np.full(100, 60.0)])
print(round(one.mean(), 1), round(one.std(), 2))
print(round(one.kurt(), 2))
print((one == 50).sum())

*Your prediction:*  

> ### ✅ Answer A7
> ```
> 50.0 10.03
> -2.02
> 0
> ```
> **The mean is 50 and not a single value equals 50.** Half the data is at 40, half at 90...
> no — half at 40 and half at 60.
>
> The **negative kurtosis** is the warning: light tails, no central peak — here, two separate
> groups. A reader told "mean 50, SD 10" pictures a bell curve centred on 50, which is the
> opposite of the truth.
>
> This is Day 20's Anscombe lesson for distributions: **always plot**.

### A8. How many earn below average?

In [ ]:
pay = pd.Series(np.r_[rng.normal(42000, 6000, 180), rng.normal(300000, 40000, 20)])
print(f"{pay.mean():,.0f}")
print(f"{pay.median():,.0f}")
print(f"{(pay < pay.mean()).mean() * 100:.0f}%")

*Your prediction:*  

> ### ✅ Answer A8
> Roughly:
> ```
> 67,000
> 42,500
> 90%
> ```
> *(Exact figures depend on the draw — run it.)*
>
> **About 90% of these people earn less than the "average" salary.** The 20 executives drag
> the mean 24,000 above the median while affecting the median by almost nothing.
>
> "The average salary here is 67,000" is arithmetically true and **completely misleading** —
> which is why every serious income statistic you will ever read is a **median**.

---
# Part B — Spot & Fix the Bug

**Teaching note:** none of these raises an error. All four produce a number that is wrong or
uninterpretable.

### B1. This should report the typical salary.

In [ ]:
print(f"Typical salary: {pay.mean():,.0f}")

*What's wrong:* 

*Your fix:*

> ### ✅ Answer B1
> **Symptom:** a confident number that describes almost nobody.
> **Cause:** the data is strongly right-skewed, and the mean chases the tail.

In [ ]:
print(f"skew: {pay.skew():.2f}   <- well above 1, so strongly skewed")
print()
print(f"mean   {pay.mean():>10,.0f}")
print(f"median {pay.median():>10,.0f}")
print(f"below the mean: {(pay < pay.mean()).mean() * 100:.0f}% of people")
print()
print("WHAT TO REPORT:")
print(f"  The median salary is {pay.median():,.0f} across {len(pay)} staff.")
print(f"  The mean is {pay.mean():,.0f}, pulled up by a small number of very")
print(f"  high earners - {(pay > pay.quantile(.9)).sum()} people above the 90th percentile.")

# The gap between mean and median IS the finding. Report both.

### B2. This should give the most common mark.

In [ ]:
m = pd.Series([1, 1, 2, 2, 3])
print("Most common:", m.mode()[0])

*What's wrong:* 

*Your fix:*

> ### ✅ Answer B2
> **Symptom:** prints `1`, implying a single clear winner.
> **Cause:** there are **two** modes, 1 and 2, and `[0]` silently picked one.

In [ ]:
m = pd.Series([1, 1, 2, 2, 3])
modes = m.mode()

if len(modes) == 1:
    print("Most common:", modes[0])
else:
    print(f"{len(modes)} values tie as most common:", modes.tolist())

# Taking [0] is fine ONLY after you have checked there is no tie.
# With categorical data, value_counts() is usually more informative anyway:
print()
print(m.value_counts())

### B3. This should describe the spread in rupees.

In [ ]:
print(f"Spread: {pay.var():,.0f}")

*What's wrong:* 

*Your fix:*

> ### ✅ Answer B3
> **Symptom:** an enormous, uninterpretable number.
> **Cause:** the **variance is in squared units** — rupees squared, which is not a thing
> anyone can picture.

In [ ]:
print(f"variance : {pay.var():>18,.0f}   rupees SQUARED - meaningless")
print(f"std dev  : {pay.std():>18,.0f}   rupees - interpretable")
print(f"IQR      : {pay.quantile(.75) - pay.quantile(.25):>18,.0f}   rupees - and robust")

# The variance is what the maths runs on; the SD is what you quote.
#
# And on data this skewed, the IQR is the better choice anyway: the SD is
# inflated by the same executives that inflated the mean.

### B4. This compares the spread of two classes.

In [ ]:
a = pd.Series(rng.normal(70, 8, 30))     # a sample of 30 students
b = pd.Series(rng.normal(70, 8, 30))
print(round(np.std(a), 3), round(b.std(), 3))

*What's wrong:* 

*Your fix:*

> ### ✅ Answer B4
> **Symptom:** the two numbers are not comparable, and the difference between them is partly
> an artefact.
> **Cause:** `np.std(a)` uses `ddof=0` while `b.std()` uses `ddof=1`. You have compared a
> population formula with a sample formula.

In [ ]:
a = pd.Series(rng.normal(70, 8, 30))
b = pd.Series(rng.normal(70, 8, 30))

print("MIXED (the bug):")
print(f"  np.std(a) = {np.std(a):.3f}   [ddof=0]")
print(f"  b.std()   = {b.std():.3f}   [ddof=1]")
print()
print("CONSISTENT (the fix) - both as samples:")
print(f"  a = {a.std(ddof=1):.3f}")
print(f"  b = {b.std(ddof=1):.3f}")
print()
print(f"the ddof choice alone shifts a by {abs(np.std(a) - a.std(ddof=1)):.3f}")

# These are samples of students, so ddof=1 throughout.
# The rule: never mix conventions inside one comparison.

---
# Part C — Write the Code

**Teaching note:** C5 is the reusable one — it is the function they carry into Module 2's
project. **C1 and C3 are a deliberate borderline case**: `marks` has skew 1.16 (over the
threshold) but a mean–median gap of only 0.20 sd (under it). The two rules disagree, and the
right answer is to say so.

### C1. Centre three ways
For `marks`, print the mean, median and mode, and say in one line which you would report.

In [ ]:
# your code here

In [ ]:
print(f"mean   {marks.mean():.1f}")
print(f"median {marks.median():.1f}")
print(f"mode   {marks.mode().tolist()}")
print()
print(f"skew   {marks.skew():.2f}")
print()
print("A BORDERLINE CASE - and worth saying so.")
print("  skew 1.16 is just OVER the usual threshold of 1  -> suggests median")
print("  but mean and median differ by only 2.5, which is 0.20 sd -> suggests mean")
print()
print("WHICH TO REPORT: both, with the n.")
print("  'Median 65.5, mean 68.0, across 10 marks.'")
print("The two rules disagree because n is small: with ten values a single 95")
print("moves the skew statistic a long way without moving the centre much.")
print("When the signals conflict, say so rather than picking one silently.")

# mean   68.0
# median 65.5
# mode   [61]
# skew    1.16

### C2. Spread four ways
For `marks`, print the range, IQR, variance and both standard deviations. Say which number
you would quote to a parent.

In [ ]:
# your code here

In [ ]:
print(f"range     {marks.max() - marks.min():8.2f}")
print(f"IQR       {marks.quantile(.75) - marks.quantile(.25):8.2f}")
print(f"variance  {marks.var():8.2f}    <- squared units")
print(f"std ddof1 {marks.std(ddof=1):8.2f}    <- sample")
print(f"std ddof0 {marks.std(ddof=0):8.2f}    <- population")
print()
print("TO A PARENT: the standard deviation, about 12 marks.")
print("It is in the same units as the marks, so 'most students are within")
print("12 marks of 68' is a sentence they can picture. The variance (150.0)")
print("is marks-SQUARED and means nothing to anyone.")

# These 10 are a SAMPLE of what this student cohort could score, so ddof=1.

### C3. Detect the skew
Write a function `is_skewed(s)` that returns True when a Series should be summarised with
the median rather than the mean. Test it on `marks` and on `pay`.

In [ ]:
# your code here

In [ ]:
def is_skewed(s, skew_limit=1.0, gap_limit=0.3):
    """True if the median is the safer summary.

    Two independent signals, either is enough:
      - |skew| above skew_limit
      - mean and median more than gap_limit standard deviations apart
    """
    s = s.dropna()
    if len(s) < 3 or s.std() == 0:
        return False
    gap = abs(s.mean() - s.median()) / s.std()
    return abs(s.skew()) > skew_limit or gap > gap_limit


for name, s in [("marks", marks), ("pay", pay)]:
    gap = abs(s.mean() - s.median()) / s.std()
    print(f"{name:6}  skew {s.skew():+6.2f}   gap {gap:5.2f} sd   "
          f"-> use {'MEDIAN' if is_skewed(s) else 'mean'}")

# marks   skew  +1.16   gap  0.20 sd   -> use MEDIAN
# pay     skew  +2.xx   gap  0.xx sd   -> use MEDIAN
#
# NOTE what happened with marks: the skew rule fired (1.16 > 1) but the gap
# rule did NOT (0.20 < 0.30). Either signal alone is enough to return True,
# which is deliberately cautious - but it means you should LOOK at which
# rule fired rather than trusting the boolean.
#
# Using TWO signals matters: a distribution can have modest skew but still
# have the mean dragged well off the median, and vice versa. On n=10 the
# skew statistic is itself unstable, which is why the two disagree here.

### C4. Compare two groups honestly
Split `pay` at the 90th percentile into 'most staff' and 'top earners'. Report each group
separately, with counts.

In [ ]:
# your code here

In [ ]:
cut = pay.quantile(.90)
most = pay[pay <= cut]
top  = pay[pay > cut]

print(f"{'group':<12} {'n':>5} {'median':>12} {'mean':>12} {'IQR':>12}")
for name, g in [("most staff", most), ("top earners", top)]:
    print(f"{name:<12} {len(g):>5} {g.median():>12,.0f} {g.mean():>12,.0f} "
          f"{g.quantile(.75) - g.quantile(.25):>12,.0f}")

print()
print(f"Combined median: {pay.median():>10,.0f}")
print(f"Combined mean  : {pay.mean():>10,.0f}   <- describes neither group")

# THE POINT: once you split, each group's mean and median are close together,
# because each group is roughly symmetric on its own. The combined mean sits
# in the gap between them, describing nobody - which is exactly the bimodal
# problem from the lecture.

### C5. An honest summary function
Write `honest_summary(s)` returning n, missing, mean, median, both standard deviations,
IQR, skew, kurtosis — and a one-word recommendation.

In [ ]:
# your code here

In [ ]:
def honest_summary(s, name="value"):
    """Centre, spread and shape - plus the recommendation describe() omits."""
    missing = int(s.isna().sum())
    s = s.dropna()
    skew = s.skew()

    out = {
        "n":        len(s),
        "missing":  missing,
        "mean":     round(s.mean(), 2),
        "median":   round(s.median(), 2),
        "std":      round(s.std(), 2),          # ddof=1, sample
        "std_pop":  round(s.std(ddof=0), 2),    # ddof=0, population
        "IQR":      round(s.quantile(.75) - s.quantile(.25), 2),
        "skew":     round(skew, 2),
        "kurtosis": round(s.kurt(), 2),
    }
    gap = abs(out["mean"] - out["median"]) / out["std"] if out["std"] else 0
    out["report"] = ("MEDIAN and IQR - skewed"
                     if abs(skew) > 1 or gap > 0.3 else "mean and std - symmetric")
    return pd.Series(out, name=name)


print(pd.concat([honest_summary(marks, "marks"),
                 honest_summary(pay, "pay")], axis=1))

# Reporting BOTH ddof values makes the choice visible rather than hidden,
# and the recommendation line is the part describe() will never give you.

---
# Part D — Challenge

**Teaching note:** D1 is the one that converts ddof from a rule into a fact they have
verified themselves.

### D1. Prove n − 1 yourself
Build a population you know the truth about. Take many small samples. Show that dividing by
`n` understates the variance, and that the shortfall is `(n−1)/n`.

In [ ]:
# your code here

In [ ]:
sim = np.random.default_rng(2026)
POP = sim.normal(100, 15, 1_000_000)
true_var = POP.var()

print(f"true population variance: {true_var:.2f}")
print()
print(f"{'n':>4} {'/n gives':>10} {'/(n-1) gives':>13} {'observed':>10} {'predicted':>10}")
print(f"{'':>4} {'':>10} {'':>13} {'ratio':>10} {'(n-1)/n':>10}")

for n in [3, 5, 10, 50, 200]:
    sm = sim.choice(POP, size=(200_000, n))
    a = sm.var(axis=1, ddof=0).mean()
    b = sm.var(axis=1, ddof=1).mean()
    print(f"{n:>4} {a:>10.2f} {b:>13.2f} {a/true_var:>10.3f} {(n-1)/n:>10.3f}")

print()
print("READ THE LAST TWO COLUMNS. The observed shortfall matches (n-1)/n at")
print("every sample size - 0.666 vs 0.667 at n=3, 0.995 vs 0.995 at n=200.")
print()
print("WHY: you measured each value's distance from the SAMPLE mean, but that")
print("mean came from those same values, so it sits closer to them than the")
print("true mean does. Your distances are too small, always, by exactly this")
print("factor. Dividing by n-1 instead of n undoes it.")
print()
print("And notice the /n column climbing towards the truth as n grows - which")
print("is why the correction matters on small samples and nobody notices it")
print("on large ones.")

### D2. The bimodal trap
The data below is two groups in disguise. Show that the mean and SD describe nobody, then
show what you would report instead.

In [ ]:
mystery = pd.Series(np.r_[rng.normal(35, 4, 500), rng.normal(75, 4, 500)])

# your code here

In [ ]:
mystery = pd.Series(np.r_[rng.normal(35, 4, 500), rng.normal(75, 4, 500)])

print("THE NAIVE SUMMARY")
print(f"  mean {mystery.mean():.1f}   median {mystery.median():.1f}   sd {mystery.std():.1f}")
print(f"  skew {mystery.skew():+.2f}  <- looks symmetric!")
print(f"  kurtosis {mystery.kurt():+.2f}  <- THE WARNING: strongly negative")
print()
within = mystery.between(mystery.mean() - 5, mystery.mean() + 5).sum()
print(f"  values within 5 of the mean: {within} of {len(mystery)}")

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.hist(mystery, bins=50, color="#3E7CB1", edgecolor="white")
ax.axvline(mystery.mean(), color="#E8674C", linestyle="--", linewidth=2,
           label=f"mean {mystery.mean():.1f}")
ax.set_title("The mean sits in the empty valley between two groups")
ax.set_xlabel("value"); ax.set_ylabel("count"); ax.legend()
plt.show()

# WHAT GIVES IT AWAY: the skew is ~0 and the mean equals the median, so
# every symmetry check passes. Only the strongly NEGATIVE kurtosis warns
# you - and the plot settles it.
print()
print("WHAT TO REPORT: split first.")
lo = mystery[mystery < 55]
hi = mystery[mystery >= 55]
print(f"  group A: n={len(lo)}  mean {lo.mean():.1f}  sd {lo.std():.1f}")
print(f"  group B: n={len(hi)}  mean {hi.mean():.1f}  sd {hi.std():.1f}")
print()
print("Two honest summaries beat one dishonest one. And the real question is")
print("now obvious: WHAT distinguishes the two groups? That is a column you")
print("have not looked at yet.")

---
## Done? Self-check

- [ ] I can say when the mean misleads and what to use instead
- [ ] I know `mode()` returns a Series, and why
- [ ] I can explain what `n − 1` corrects for
- [ ] I know which `ddof` to use for a sample and for a population
- [ ] I know why we quote the SD and not the variance
- [ ] I know what a negative kurtosis warns me about
- [ ] I reported `n` with every summary

### Homework
1. Reproduce the n−1 simulation and try n = 3, 10 and 50.
2. Find a real dataset where the mean and median disagree, and explain why.
3. Run `honest_summary` on three columns of your Day 21 project data.

### Next class — Topic 2.4: The normal distribution
The bell curve, the 68–95–99.7 rule, and z-scores.

---
*Prepared by Srinivasa Sai Chava · Boston University*

---
## Wrap-up — running the last 5 minutes

Three cold-call questions:

1. *"NumPy std and pandas std on the same data — same answer?"* → no, `ddof` differs.
2. *"What does n − 1 correct for?"* → the sample mean sits too close to its own data.
3. *"Skew near zero means symmetric?"* → not necessarily — check the kurtosis and plot.

**Common misconceptions to watch for today**

| Misconception | Correction |
|---|---|
| "The average is the typical value" | Only when the data is roughly symmetric |
| "ddof is a technicality" | It changes the number, and the two libraries disagree |
| "Bigger data means use ddof=0" | The test is whether anything is **unseen**, not size |
| "Positive skew means most values are high" | It means the **tail** is on the right |
| "Skew ≈ 0 means bell-shaped" | Two symmetric humps also score 0 |
| "Variance and SD are interchangeable" | Variance is in squared units; quote the SD |

**D1 is the centrepiece.** Students have been told "pandas uses n−1" since Day 14 and have
taken it on trust. Having them generate the table themselves — 0.666 against a predicted
0.667, 0.995 against 0.995 — converts a rule into something they have verified. Several will
be visibly surprised that it works out so exactly.

**D2 is the subtle one.** The bimodal data has skew ≈ 0 and mean = median, so every symmetry
check they know will pass. Only the strongly negative kurtosis warns them, and only the plot
settles it. Point out that the useful next question — *what distinguishes the two groups?* —
is about a column they have not looked at.

**A5 is worth discussing aloud.** Most students assume `ddof=0` is for small data and
`ddof=1` for large, which is backwards reasoning. The test is whether anything is unseen.

**Homework given:** reproduce the simulation at three sample sizes; find a real
mean-vs-median disagreement; run `honest_summary` on their Day 21 data.

**Next session:** Topic 2.4 — the normal distribution, where today's standard deviation stops
being a description and becomes a ruler.